In [0]:
# =====================================================
# Notebook : 06_build_dim_client_scd2
# Objectif : Construire dim_client avec historisation
#            complète (SCD Type 2) à partir de Silver
# Résout   : l'exigence réglementaire "état à une date passée"
# =====================================================

from pyspark.sql.functions import (
    col, lit, current_date, current_timestamp, 
    monotonically_increasing_id, row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "banking_lakehouse"
TABLE_CLIENTS_SILVER = f"{CATALOG}.silver.clients"
TABLE_DIM_CLIENT_SCD2 = f"{CATALOG}.gold.dim_client"

# Colonnes qu'on va suivre pour détecter un changement (attributs "trackés" en SCD2)
TRACKED_COLUMNS = ["CreditScore", "Balance", "IsActiveMember", "Geography"]

print("✅ Configuration SCD2 chargée")
print(f"Source Silver : {TABLE_CLIENTS_SILVER}")
print(f"Cible Gold    : {TABLE_DIM_CLIENT_SCD2}")
print(f"Colonnes trackées pour historisation : {TRACKED_COLUMNS}")

In [0]:
# =====================================================
# Initialisation SCD2 - Premier chargement
# Toutes les lignes deviennent "version 1", actuelles
# =====================================================

df_silver_clients = spark.table(TABLE_CLIENTS_SILVER)

table_exists = spark.catalog.tableExists(TABLE_DIM_CLIENT_SCD2)

if not table_exists:
    print(f"🆕 Initialisation de {TABLE_DIM_CLIENT_SCD2}")
    
    df_dim_client_init = (
        df_silver_clients
        # --- Surrogate Key technique (unique par VERSION, pas par client) ---
        .withColumn("client_sk", monotonically_increasing_id())
        
        # --- Colonnes SCD Type 2 ---
        .withColumn("valid_from", current_date())
        .withColumn("valid_to", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        
        # --- Métadonnée de traçabilité ---
        .withColumn("_scd_created_at", current_timestamp())
        
        # On sélectionne les colonnes utiles (on retire les flags DQ, déjà validés en Silver)
        .select(
            "client_sk", "CustomerId", "Surname", "CreditScore", "Geography",
            "Gender", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard",
            "IsActiveMember", "EstimatedSalary", "Exited", "Country_Risk_Level",
            "valid_from", "valid_to", "is_current", "_scd_created_at"
        )
    )
    
    df_dim_client_init.write.format("delta").saveAsTable(TABLE_DIM_CLIENT_SCD2)
    
    nb_lignes = spark.table(TABLE_DIM_CLIENT_SCD2).count()
    print(f"✅ dim_client initialisée avec {nb_lignes} lignes (toutes version 1, is_current=True)")
else:
    print(f"ℹ️ La table {TABLE_DIM_CLIENT_SCD2} existe déjà — pas de réinitialisation")
    print("   (utilise la Cellule 3 pour appliquer un SCD2 incrémental)")

# --- Vérification ---
print("\n📋 Aperçu de dim_client :")
spark.table(TABLE_DIM_CLIENT_SCD2).select(
    "client_sk", "CustomerId", "Surname", "CreditScore", "Balance",
    "valid_from", "valid_to", "is_current"
).show(5)

In [0]:
# =====================================================
# TEST SCD2 : simuler un NOUVEAU changement métier
# Scénario : le client 15634602 voit son statut
# IsActiveMember passer de 1 à 0 (devient inactif)
# =====================================================

# On récupère l'état actuel en Silver pour ce client
df_client_test = spark.table(TABLE_CLIENTS_SILVER).filter(col("CustomerId") == 15634602)

print("📋 État actuel en Silver (avant le nouveau changement) :")
df_client_test.select("CustomerId", "Surname", "CreditScore", "Balance", "IsActiveMember").show()

# On simule le changement métier : le client devient inactif
df_nouvelle_version_silver = (
    df_client_test
    .withColumn("IsActiveMember", lit(0))  # ancien : 1 -> nouveau : 0
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_silver_processed_at", current_timestamp())
)

print("\n📋 Nouvelle version simulée :")
df_nouvelle_version_silver.select("CustomerId", "Surname", "CreditScore", "Balance", "IsActiveMember").show()

# --- On met à jour Silver avec cette nouvelle donnée (comme un vrai flux amont) ---
delta_silver = DeltaTable.forName(spark, TABLE_CLIENTS_SILVER)

(
    delta_silver.alias("target")
    .merge(
        df_nouvelle_version_silver.alias("source"),
        "target.CustomerId = source.CustomerId"
    )
    .whenMatchedUpdate(
        condition="source._ingestion_timestamp > target._ingestion_timestamp",
        set={
            "IsActiveMember": "source.IsActiveMember",
            "_ingestion_timestamp": "source._ingestion_timestamp",
            "_silver_processed_at": "source._silver_processed_at"
        }
    )
    .execute()
)

print("\n✅ Silver mis à jour avec le nouveau changement métier")

In [0]:
# =====================================================
# SCD TYPE 2 - ÉTAPE A : Détection des changements
# On compare la version "is_current=True" de dim_client
# avec les données Silver les plus récentes
# =====================================================

df_dim_current = spark.table(TABLE_DIM_CLIENT_SCD2).filter(col("is_current") == True)
df_silver_latest = spark.table(TABLE_CLIENTS_SILVER)

# Jointure pour détecter les changements sur les colonnes trackées
df_comparison = (
    df_silver_latest.alias("src")
    .join(
        df_dim_current.alias("dim"),
        col("src.CustomerId") == col("dim.CustomerId"),
        "left"
    )
)

# Construction dynamique de la condition de changement
change_condition = None
for tracked_col in TRACKED_COLUMNS:
    condition = col(f"src.{tracked_col}") != col(f"dim.{tracked_col}")
    change_condition = condition if change_condition is None else (change_condition | condition)

# --- Cas 1 : Clients EXISTANTS avec changement détecté ---
df_changed_clients = (
    df_comparison
    .filter(col("dim.CustomerId").isNotNull())  # le client existe déjà en dim
    .filter(change_condition)                    # ET au moins une colonne trackée a changé
    .select("src.*")  # on garde les valeurs SOURCE (les plus récentes)
)

# --- Cas 2 : NOUVEAUX clients (n'existent pas encore en dim) ---
df_new_clients = (
    df_comparison
    .filter(col("dim.CustomerId").isNull())
    .select("src.*")
)

nb_changed = df_changed_clients.count()
nb_new = df_new_clients.count()

print(f"🔎 Clients avec changement détecté : {nb_changed}")
print(f"🆕 Nouveaux clients détectés        : {nb_new}")

if nb_changed > 0:
    print("\n📋 Détail des changements :")
    df_changed_clients.select("CustomerId", "Surname", "CreditScore", "Balance", "IsActiveMember", "Geography").show()

In [0]:
# =====================================================
# SCD TYPE 2 - ÉTAPE B : Clôture des anciennes versions
# Pour les clients qui ont changé : is_current=False, 
# valid_to=aujourd'hui
# =====================================================

delta_dim_client = DeltaTable.forName(spark, TABLE_DIM_CLIENT_SCD2)

if nb_changed > 0:
    # Liste des CustomerId ayant changé (pour cibler précisément le UPDATE)
    changed_customer_ids = [row.CustomerId for row in df_changed_clients.select("CustomerId").collect()]
    
    print(f"🔒 Clôture de {len(changed_customer_ids)} version(s) actuelle(s)...")
    
    (
        delta_dim_client.update(
            condition=(col("CustomerId").isin(changed_customer_ids)) & (col("is_current") == True),
            set={
                "is_current": lit(False),
                "valid_to": current_date()
            }
        )
    )
    
    print("✅ Anciennes versions clôturées")
    
    # Vérification
    print("\n📋 Vérification - historique du client 15634602 après clôture :")
    spark.table(TABLE_DIM_CLIENT_SCD2).filter(col("CustomerId") == 15634602).select(
        "client_sk", "CustomerId", "CreditScore", "IsActiveMember", "valid_from", "valid_to", "is_current"
    ).show()
else:
    print("ℹ️ Aucun changement à clôturer")

In [0]:
# =====================================================
# SCD TYPE 2 - ÉTAPE C : Insertion de la nouvelle version
# Nouvelle ligne avec les valeurs à jour, is_current=True
# =====================================================

if nb_changed > 0 or nb_new > 0:
    
    # On récupère le max actuel de client_sk pour générer des SK uniques et croissants
    max_sk = spark.table(TABLE_DIM_CLIENT_SCD2).agg({"client_sk": "max"}).collect()[0][0]
    
    # Union des clients changés + nouveaux clients (les deux cas nécessitent un INSERT)
    df_rows_to_insert = df_changed_clients.unionByName(df_new_clients)
    
    df_new_versions = (
        df_rows_to_insert
        # Nouvelle surrogate key, en continuité avec l'existant
        .withColumn("client_sk", monotonically_increasing_id() + lit(max_sk + 1))
        .withColumn("valid_from", current_date())
        .withColumn("valid_to", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("_scd_created_at", current_timestamp())
        .select(
            "client_sk", "CustomerId", "Surname", "CreditScore", "Geography",
            "Gender", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard",
            "IsActiveMember", "EstimatedSalary", "Exited", "Country_Risk_Level",
            "valid_from", "valid_to", "is_current", "_scd_created_at"
        )
    )
    
    df_new_versions.write.format("delta").mode("append").saveAsTable(TABLE_DIM_CLIENT_SCD2)
    
    print(f"✅ {df_new_versions.count()} nouvelle(s) version(s) insérée(s)")
else:
    print("ℹ️ Rien à insérer")

# --- Vérification finale : historique COMPLET du client 15634602 ---
print("\n📋 HISTORIQUE COMPLET du client 15634602 :")
spark.table(TABLE_DIM_CLIENT_SCD2).filter(col("CustomerId") == 15634602).select(
    "client_sk", "CustomerId", "CreditScore", "IsActiveMember", "valid_from", "valid_to", "is_current"
).orderBy("valid_from").show()

# --- Comptage global ---
nb_total_dim = spark.table(TABLE_DIM_CLIENT_SCD2).count()
nb_current = spark.table(TABLE_DIM_CLIENT_SCD2).filter(col("is_current") == True).count()
print(f"\n📊 Total lignes dans dim_client (toutes versions) : {nb_total_dim}")
print(f"📊 Lignes 'actuelles' (is_current=True)              : {nb_current}")

In [0]:
%sql
-- Preuve : répondre à "quel était l'état du client à une date donnée ?"
SELECT client_sk, CustomerId, CreditScore, IsActiveMember, valid_from, valid_to, is_current
FROM banking_lakehouse.gold.dim_client
WHERE CustomerId = 15634602
ORDER BY valid_from;

In [0]:
%sql
SELECT CreditScore, IsActiveMember 
FROM banking_lakehouse.gold.dim_client
WHERE CustomerId = 15634602
AND '2026-08-23' BETWEEN valid_from AND COALESCE(valid_to, '9999-12-31')